# PODCAST Benchmark — Result Matrix Generator

Generates:
1. **Primary metric result matrix** (Model x Task) — saved as `results/primary_metric_matrix.csv`
2. **Per-task full metric CSVs** — saved as `results/{task_name}_full_metrics.csv`

Each output has multiple hierarchy levels:
- **Raw**: lag x fold granularity
- **Fold-averaged**: averaged across folds (per lag)
- **Lag-averaged**: averaged across lags (per fold)
- **Fully-averaged**: single value per model (averaged across both folds and lags)
- **Best-lag**: performance at the best lag (by val primary metric)

In [40]:
import os
import glob
import re
import yaml
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

## Configuration

In [41]:
REPO_ROOT = Path(os.path.abspath('')).parent
FOUNDATION_RESULTS_DIR = REPO_ROOT / 'results' / 'foundation_models'
BASELINE_RESULTS_DIR = REPO_ROOT / 'baseline-results'

# Task directory name -> formal task_name mapping
TASK_DIR_TO_NAME = {
    'content_noncontent': 'content_noncontent_task',
    'sentence_onset': 'sentence_onset_task',
    'iu_boundary': 'iu_boundary_task',
    'gpt_surprise': 'gpt_surprise_task',
    'gpt_surprise_multiclass': 'gpt_surprise_multiclass_task',
    'pos': 'pos_task',
    'word_embedding': 'word_embedding_decoding_task',
    'whisper_embedding': 'whisper_embedding_decoding_task',
    'volume_level': 'volume_level_decoding_task',
    'llm_decoding': 'llm_decoding_task',
    'llm_embedding_pretraining': 'llm_embedding_pretraining_task',
}
TASK_NAME_TO_DIR = {v: k for k, v in TASK_DIR_TO_NAME.items()}

# Primary metric per task (test split)
PRIMARY_METRIC = {
    'content_noncontent_task': 'roc_auc',
    'sentence_onset_task': 'roc_auc',
    'iu_boundary_task': 'roc_auc',
    'gpt_surprise_task': 'corr',
    'gpt_surprise_multiclass_task': 'roc_auc_multiclass',
    'pos_task': 'roc_auc_multiclass',
    'word_embedding_decoding_task': 'pairwise_accuracy',
    'whisper_embedding_decoding_task': 'pairwise_accuracy',
    'volume_level_decoding_task': 'corr',
    'llm_decoding_task': 'perplexity_llm',
    'llm_embedding_pretraining_task': 'mse',
}

# For best-lag selection: higher is better for most, lower for these
LOWER_IS_BETTER = {'perplexity_llm', 'mse', 'cross_entropy', 'bce_with_logits', 'cosine_dist', 'nll_embedding'}

# Subject level to use for foundation models
SUBJECT_LEVEL = 'supersubject'

# Foundation model names
FOUNDATION_MODELS = ['brainbert', 'diver', 'popt']

# Output directory — scoped by subject level
OUTPUT_DIR = Path(os.path.abspath('')) / 'results' / SUBJECT_LEVEL
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Repo root: {REPO_ROOT}')
print(f'Foundation results: {FOUNDATION_RESULTS_DIR}')
print(f'Baseline results: {BASELINE_RESULTS_DIR}')
print(f'Subject level: {SUBJECT_LEVEL}')
print(f'Output dir: {OUTPUT_DIR}')

Repo root: /pscratch/sd/a/ahhyun/EcoGFound/PODCAST/podcast-benchmark
Foundation results: /pscratch/sd/a/ahhyun/EcoGFound/PODCAST/podcast-benchmark/results/foundation_models
Baseline results: /pscratch/sd/a/ahhyun/EcoGFound/PODCAST/podcast-benchmark/baseline-results
Subject level: supersubject
Output dir: /pscratch/sd/a/ahhyun/EcoGFound/PODCAST/podcast-benchmark/notebooks/results/supersubject


## Baseline Configuration

Map baseline result directories to their canonical task names. Edit this dict to select which baseline run to use per task.

In [42]:
# Baseline dir name -> task_name mapping
# One baseline per task. Edit to choose different baseline runs.
BASELINE_RUNS = {
    'ensemble_model_10_gpt2_2026-02-17-19-54-55': 'word_embedding_decoding_task',
    'neural_conv_whisper_embedding_2026-02-17-19-25-13': 'whisper_embedding_decoding_task',
    'gpt_surprise_2025-12-19-00-18-44': 'gpt_surprise_task',
    'gpt_surprise_2025-12-19-00-18-43': 'gpt_surprise_multiclass_task',
    'content_noncontent_task_sig_elecs_mlp_early_stop_roc_2025-12-19-00-34-17': 'content_noncontent_task',
    'sentence_onset_lr_2025-12-19-00-18-44': 'sentence_onset_task',
    'iu_boundary_lr_2026-02-26-09-46-13': 'iu_boundary_task',
    'pos_task_sig_elecs_without_other_classes_2025-12-19-00-34-17': 'pos_task',
    'volume_level_torch_ridge_2025-12-19-00-42-42': 'volume_level_decoding_task',
    'llm_token_finetune_2025-12-26-12-44-36': 'llm_decoding_task',
    # 'llm_embedding_pretraining' has no baseline run yet
}

print(f'Baseline tasks covered: {len(BASELINE_RUNS)}')
print(f'Missing baseline tasks: {set(PRIMARY_METRIC.keys()) - set(BASELINE_RUNS.values())}')

Baseline tasks covered: 10
Missing baseline tasks: {'llm_embedding_pretraining_task'}


## Data Loading Utilities

In [43]:
def is_valid_run(run_dir, required_epochs=100):
    """Check config.yml to filter out debug runs (epochs != required_epochs)."""
    cfg_path = Path(run_dir) / 'config.yml'
    if not cfg_path.exists():
        return False
    try:
        with open(cfg_path) as f:
            cfg = yaml.safe_load(f)
        return cfg.get('training_params', {}).get('epochs') == required_epochs
    except Exception:
        return False


def get_latest_run(run_dirs, filter_epochs=100):
    """Pick the most recent valid run directory (epochs == filter_epochs).
    Set filter_epochs=None to skip the config check (e.g. for baselines).
    """
    if not run_dirs:
        return None
    if filter_epochs is not None:
        run_dirs = [d for d in run_dirs if is_valid_run(d, required_epochs=filter_epochs)]
    if not run_dirs:
        return None
    # Sort by modification time (most recent last)
    return sorted(run_dirs, key=lambda p: os.path.getmtime(p))[-1]


def parse_fold_columns(df):
    """Extract fold numbers from column names like test_corr_fold_1."""
    fold_nums = set()
    for col in df.columns:
        m = re.search(r'_fold_(\d+)$', col)
        if m:
            fold_nums.add(int(m.group(1)))
    return sorted(fold_nums)


def get_metrics_from_columns(df, split='test'):
    """Extract unique metric names for a given split from column names."""
    metrics = set()
    for col in df.columns:
        if col.startswith(f'{split}_') and col.endswith('_mean'):
            metric = col[len(f'{split}_'):-len('_mean')]
            metrics.add(metric)
    return sorted(metrics)


def load_result(csv_path):
    """Load a lag_performance.csv and return the DataFrame."""
    if not os.path.exists(csv_path):
        return None
    df = pd.read_csv(csv_path)
    if df.empty:
        return None
    return df


def find_foundation_results(model, task_dir, subject_level=SUBJECT_LEVEL):
    """Find all valid runs for a foundation model + task, pick latest per lag,
    and return a single concatenated DataFrame with all lags.
    """
    task_path = FOUNDATION_RESULTS_DIR / model / task_dir / subject_level
    if not task_path.exists():
        return None, None
    run_dirs = [p for p in task_path.iterdir() if p.is_dir()]
    # Filter to valid runs (epochs == 100)
    run_dirs = [d for d in run_dirs if is_valid_run(d, required_epochs=100)]
    if not run_dirs:
        return None, None

    # Load each run's lag_performance.csv, track lag -> (mtime, df, path)
    best_per_lag = {}
    for d in run_dirs:
        csv_path = d / 'lag_performance.csv'
        if not csv_path.exists():
            continue
        df = pd.read_csv(csv_path)
        if df.empty:
            continue
        lag = df['lags'].iloc[0]
        mtime = os.path.getmtime(d)
        if lag not in best_per_lag or mtime > best_per_lag[lag][0]:
            best_per_lag[lag] = (mtime, df, str(csv_path))

    if not best_per_lag:
        return None, None

    # Concat all lags, sorted
    combined = pd.concat(
        [info[1] for lag, info in sorted(best_per_lag.items())],
        ignore_index=True
    )
    paths = [info[2] for lag, info in sorted(best_per_lag.items())]
    return combined, paths


print('Utilities loaded.')

Utilities loaded.


## Load All Results

In [44]:
# Structure: results_raw[model_name][task_name] = DataFrame
results_raw = defaultdict(dict)
result_paths = {}  # for reference

# Load foundation model results
for model in FOUNDATION_MODELS:
    for task_dir, task_name in TASK_DIR_TO_NAME.items():
        combined_df, paths = find_foundation_results(model, task_dir)
        if combined_df is not None:
            results_raw[model][task_name] = combined_df
            result_paths[(model, task_name)] = '; '.join(paths)

# Load baseline results
for baseline_dir, task_name in BASELINE_RUNS.items():
    csv_path = BASELINE_RESULTS_DIR / baseline_dir / 'lag_performance.csv'
    if csv_path.exists():
        df = load_result(csv_path)
        if df is not None:
            results_raw['baseline'][task_name] = df
            result_paths[('baseline', task_name)] = str(csv_path)

# Summary
print('Results loaded:')
for model in ['baseline'] + FOUNDATION_MODELS:
    tasks = sorted(results_raw.get(model, {}).keys())
    print(f'  {model}: {len(tasks)} tasks — {tasks}')

Results loaded:
  baseline: 10 tasks — ['content_noncontent_task', 'gpt_surprise_multiclass_task', 'gpt_surprise_task', 'iu_boundary_task', 'llm_decoding_task', 'pos_task', 'sentence_onset_task', 'volume_level_decoding_task', 'whisper_embedding_decoding_task', 'word_embedding_decoding_task']
  brainbert: 8 tasks — ['content_noncontent_task', 'gpt_surprise_multiclass_task', 'gpt_surprise_task', 'iu_boundary_task', 'llm_embedding_pretraining_task', 'pos_task', 'sentence_onset_task', 'whisper_embedding_decoding_task']
  diver: 8 tasks — ['content_noncontent_task', 'gpt_surprise_multiclass_task', 'gpt_surprise_task', 'iu_boundary_task', 'llm_embedding_pretraining_task', 'pos_task', 'sentence_onset_task', 'whisper_embedding_decoding_task']
  popt: 8 tasks — ['content_noncontent_task', 'gpt_surprise_multiclass_task', 'gpt_surprise_task', 'iu_boundary_task', 'llm_embedding_pretraining_task', 'pos_task', 'sentence_onset_task', 'whisper_embedding_decoding_task']


## Build Multi-Level Hierarchical DataFrames

For each (model, task), we produce:
1. **Raw (lag x fold)**: Every lag-fold combination
2. **Fold-averaged**: Mean/std across folds per lag (already in the CSV as `_mean`/`_std`)
3. **Lag-averaged**: Mean across lags per fold
4. **Fully-averaged**: Single value (mean across all lags and folds)
5. **Best-lag**: Values at the lag with best validation primary metric

In [45]:
def build_raw_lag_fold(df, split='test'):
    """Level 1: Expand to raw lag x fold rows.
    Returns DataFrame with columns: lag, fold, metric1, metric2, ...
    """
    folds = parse_fold_columns(df)
    metrics = get_metrics_from_columns(df, split)
    if not folds or not metrics:
        return None

    rows = []
    for _, row in df.iterrows():
        lag = row['lags']
        for fold in folds:
            entry = {'lag': lag, 'fold': fold}
            for metric in metrics:
                col = f'{split}_{metric}_fold_{fold}'
                if col in df.columns:
                    entry[metric] = row[col]
            rows.append(entry)
    return pd.DataFrame(rows)


def build_fold_averaged(df, split='test'):
    """Level 2: Mean and std across folds, per lag.
    Returns DataFrame with columns: lag, metric1_mean, metric1_std, ...
    """
    metrics = get_metrics_from_columns(df, split)
    if not metrics:
        return None

    rows = []
    for _, row in df.iterrows():
        entry = {'lag': row['lags']}
        for metric in metrics:
            mean_col = f'{split}_{metric}_mean'
            std_col = f'{split}_{metric}_std'
            if mean_col in df.columns:
                entry[f'{metric}_mean'] = row[mean_col]
            if std_col in df.columns:
                entry[f'{metric}_std'] = row[std_col]
        rows.append(entry)
    return pd.DataFrame(rows)


def build_lag_averaged(df, split='test'):
    """Level 3: Mean across lags, per fold.
    Returns DataFrame with columns: fold, metric1, metric2, ...
    """
    raw = build_raw_lag_fold(df, split)
    if raw is None:
        return None
    return raw.groupby('fold').mean(numeric_only=True).drop(columns=['lag'], errors='ignore').reset_index()


def build_fully_averaged(df, split='test'):
    """Level 4: Single value — mean across all lags and folds.
    Returns a Series with metric names as index.
    """
    raw = build_raw_lag_fold(df, split)
    if raw is None:
        return None
    numeric_cols = [c for c in raw.columns if c not in ('lag', 'fold')]
    means = raw[numeric_cols].mean()
    stds = raw[numeric_cols].std()
    result = pd.DataFrame({'mean': means, 'std': stds})
    return result


def build_best_lag(df, task_name, split='test'):
    """Level 5: Performance at the best lag (selected by val primary metric).
    Returns a Series with metric_mean and metric_std.
    """
    primary = PRIMARY_METRIC.get(task_name)
    if primary is None:
        return None

    val_col = f'val_{primary}_mean'
    if val_col not in df.columns:
        # Fallback: use test split for selection
        val_col = f'test_{primary}_mean'
        if val_col not in df.columns:
            return None

    if primary in LOWER_IS_BETTER:
        best_idx = df[val_col].idxmin()
    else:
        best_idx = df[val_col].idxmax()

    best_row = df.iloc[best_idx]
    best_lag = best_row['lags']

    metrics = get_metrics_from_columns(df, split)
    result = {'best_lag': best_lag}
    for metric in metrics:
        mean_col = f'{split}_{metric}_mean'
        std_col = f'{split}_{metric}_std'
        if mean_col in df.columns:
            result[f'{metric}_mean'] = best_row[mean_col]
        if std_col in df.columns:
            result[f'{metric}_std'] = best_row[std_col]
    return pd.Series(result)


print('Hierarchy builders ready.')

Hierarchy builders ready.


## Generate Primary Metric Matrix (Model x Task)

Multiple hierarchy levels for the primary metric only.

In [46]:
all_models = ['baseline'] + FOUNDATION_MODELS
all_tasks = sorted(PRIMARY_METRIC.keys())

# --- Level: Best-lag primary metric (main result matrix) ---
matrix_best_lag = pd.DataFrame(index=all_models, columns=all_tasks, dtype=float)
matrix_best_lag_std = pd.DataFrame(index=all_models, columns=all_tasks, dtype=float)
matrix_best_lag_which = pd.DataFrame(index=all_models, columns=all_tasks)  # which lag was best

# --- Level: Fully-averaged primary metric ---
matrix_avg_all = pd.DataFrame(index=all_models, columns=all_tasks, dtype=float)
matrix_avg_all_std = pd.DataFrame(index=all_models, columns=all_tasks, dtype=float)

for model in all_models:
    for task_name in all_tasks:
        df = results_raw.get(model, {}).get(task_name)
        if df is None:
            continue

        primary = PRIMARY_METRIC[task_name]

        # Best-lag
        best = build_best_lag(df, task_name, split='test')
        if best is not None:
            matrix_best_lag.loc[model, task_name] = best.get(f'{primary}_mean', np.nan)
            matrix_best_lag_std.loc[model, task_name] = best.get(f'{primary}_std', np.nan)
            matrix_best_lag_which.loc[model, task_name] = best.get('best_lag', np.nan)

        # Fully averaged
        full_avg = build_fully_averaged(df, split='test')
        if full_avg is not None and primary in full_avg.index:
            matrix_avg_all.loc[model, task_name] = full_avg.loc[primary, 'mean']
            matrix_avg_all_std.loc[model, task_name] = full_avg.loc[primary, 'std']

print('Primary metric matrices built.')

Primary metric matrices built.


In [47]:
# Display: Best-lag primary metric (mean +/- std)
print('=== Primary Metric Matrix — Best Lag (test split) ===')
print(f'(selected by best validation primary metric per model/task)\n')

# Combined display: mean ± std
display_best = matrix_best_lag.copy()
for model in all_models:
    for task in all_tasks:
        m = matrix_best_lag.loc[model, task]
        s = matrix_best_lag_std.loc[model, task]
        lag = matrix_best_lag_which.loc[model, task]
        if pd.notna(m):
            display_best.loc[model, task] = f'{m:.4f} ± {s:.4f} (lag={lag})'
        else:
            display_best.loc[model, task] = '—'

display(display_best)

=== Primary Metric Matrix — Best Lag (test split) ===
(selected by best validation primary metric per model/task)



/tmp/ipykernel_779368/2436055110.py:13: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.5900 ± 0.0176 (lag=200.0)' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  display_best.loc[model, task] = f'{m:.4f} ± {s:.4f} (lag={lag})'
/tmp/ipykernel_779368/2436055110.py:13: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.5287 ± 0.0220 (lag=400.0)' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  display_best.loc[model, task] = f'{m:.4f} ± {s:.4f} (lag={lag})'
/tmp/ipykernel_779368/2436055110.py:13: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.0591 ± 0.0520 (lag=400.0)' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  dis

,content_noncontent_task,gpt_surprise_multiclass_task,gpt_surprise_task,iu_boundary_task,llm_decoding_task,llm_embedding_pretraining_task,pos_task,sentence_onset_task,volume_level_decoding_task,whisper_embedding_decoding_task,word_embedding_decoding_task
baseline,0.5900 ± 0.0176 (lag=200.0),0.5287 ± 0.0220 (lag=400.0),0.0591 ± 0.0520 (lag=400.0),0.5010 ± 0.0236 (lag=-800.0),—,—,0.5269 ± 0.0104 (lag=200.0),0.8800 ± 0.0435 (lag=0.0),0.4476 ± 0.0499 (lag=200.0),0.7074 ± 0.0190 (lag=400.0),0.5116 ± 0.0037 (lag=400.0)
brainbert,0.5248 ± 0.0215 (lag=-250.0),0.4858 ± 0.0051 (lag=0.0),0.0149 ± 0.0022 (lag=0.0),0.7192 ± 0.0248 (lag=0.0),—,0.0640 ± 0.0029 (lag=500.0),0.4940 ± 0.0225 (lag=0.0),0.4726 ± 0.0103 (lag=-250.0),—,0.7429 ± 0.0107 (lag=250.0),—
diver,0.5329 ± 0.0303 (lag=0.0),0.5075 ± 0.0085 (lag=250.0),0.0561 ± 0.0120 (lag=-250.0),0.6205 ± 0.0938 (lag=250.0),—,0.0113 ± 0.0001 (lag=500.0),0.5339 ± 0.0401 (lag=0.0),0.6039 ± 0.0196 (lag=-250.0),—,0.7330 ± 0.0011 (lag=500.0),—
popt,0.5270 ± 0.0382 (lag=-250.0),0.4934 ± 0.0151 (lag=500.0),-0.0018 ± 0.0270 (lag=500.0),0.5481 ± 0.0551 (lag=0.0),—,0.0113 ± 0.0002 (lag=0.0),0.5405 ± 0.0014 (lag=-250.0),0.5337 ± 0.0486 (lag=500.0),—,0.5989 ± 0.0062 (lag=250.0),—


In [48]:
# Display: Fully-averaged primary metric
print('=== Primary Metric Matrix — Averaged Across All Lags & Folds (test split) ===')
print()

display_avg = matrix_avg_all.copy()
for model in all_models:
    for task in all_tasks:
        m = matrix_avg_all.loc[model, task]
        s = matrix_avg_all_std.loc[model, task]
        if pd.notna(m):
            display_avg.loc[model, task] = f'{m:.4f} ± {s:.4f}'
        else:
            display_avg.loc[model, task] = '—'

display(display_avg)

=== Primary Metric Matrix — Averaged Across All Lags & Folds (test split) ===



/tmp/ipykernel_779368/1267711113.py:13: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '—' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  display_avg.loc[model, task] = '—'
/tmp/ipykernel_779368/1267711113.py:13: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '—' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  display_avg.loc[model, task] = '—'
/tmp/ipykernel_779368/1267711113.py:13: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '—' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  display_avg.loc[model, task] = '—'
/tmp/ipykernel_779368/1267711113.py:11: FutureWarning: Setting an item of incompatible dtype is deprecated

,content_noncontent_task,gpt_surprise_multiclass_task,gpt_surprise_task,iu_boundary_task,llm_decoding_task,llm_embedding_pretraining_task,pos_task,sentence_onset_task,volume_level_decoding_task,whisper_embedding_decoding_task,word_embedding_decoding_task
baseline,—,—,—,0.5016 ± 0.0533,—,—,—,—,—,0.6355 ± 0.0557,0.5025 ± 0.0065
brainbert,0.4993 ± 0.0329,0.5091 ± 0.0174,0.0040 ± 0.0300,0.6646 ± 0.0468,—,0.0671 ± 0.0028,0.4971 ± 0.0156,0.4901 ± 0.0368,—,0.6627 ± 0.0676,—
diver,0.5178 ± 0.0254,0.5063 ± 0.0180,0.0378 ± 0.0322,0.5954 ± 0.0998,—,0.0114 ± 0.0002,0.5206 ± 0.0294,0.5758 ± 0.0240,—,0.6826 ± 0.0452,—
popt,0.5181 ± 0.0292,0.5002 ± 0.0145,0.0033 ± 0.0212,0.5480 ± 0.0651,—,0.0113 ± 0.0002,0.5146 ± 0.0245,0.5226 ± 0.0412,—,0.6002 ± 0.0162,—


## Save Primary Metric Matrices

In [49]:
# Save numeric matrices as CSV with MultiIndex columns
# Combine best-lag and avg-all into one file with hierarchy

primary_dfs = {}

# Best-lag level
best_lag_combined = pd.concat(
    [matrix_best_lag.add_suffix('_mean'), matrix_best_lag_std.add_suffix('_std')],
    axis=1
)
# Reorder columns: group by task
ordered_cols = []
for task in all_tasks:
    ordered_cols.append(f'{task}_mean')
    ordered_cols.append(f'{task}_std')
best_lag_combined = best_lag_combined[[c for c in ordered_cols if c in best_lag_combined.columns]]
best_lag_combined.index.name = 'model'
primary_dfs['best_lag'] = best_lag_combined

# Fully averaged level
avg_all_combined = pd.concat(
    [matrix_avg_all.add_suffix('_mean'), matrix_avg_all_std.add_suffix('_std')],
    axis=1
)
avg_all_combined = avg_all_combined[[c for c in ordered_cols if c in avg_all_combined.columns]]
avg_all_combined.index.name = 'model'
primary_dfs['avg_all_lags_folds'] = avg_all_combined

# Which lag was best
matrix_best_lag_which.index.name = 'model'
primary_dfs['best_lag_selected'] = matrix_best_lag_which

# Save each level as a separate sheet-like CSV
for level_name, level_df in primary_dfs.items():
    out_path = OUTPUT_DIR / f'primary_metric_matrix_{level_name}.csv'
    level_df.to_csv(out_path)
    print(f'Saved: {out_path}')

# Also save a combined Excel-like CSV with level as first column
combined_rows = []
for level_name, level_df in primary_dfs.items():
    temp = level_df.copy()
    temp.insert(0, 'level', level_name)
    combined_rows.append(temp)
combined = pd.concat(combined_rows)
combined.to_csv(OUTPUT_DIR / 'primary_metric_matrix_all_levels.csv')
print(f'\nSaved combined: {OUTPUT_DIR / "primary_metric_matrix_all_levels.csv"}')

Saved: /pscratch/sd/a/ahhyun/EcoGFound/PODCAST/podcast-benchmark/notebooks/results/supersubject/primary_metric_matrix_best_lag.csv
Saved: /pscratch/sd/a/ahhyun/EcoGFound/PODCAST/podcast-benchmark/notebooks/results/supersubject/primary_metric_matrix_avg_all_lags_folds.csv
Saved: /pscratch/sd/a/ahhyun/EcoGFound/PODCAST/podcast-benchmark/notebooks/results/supersubject/primary_metric_matrix_best_lag_selected.csv

Saved combined: /pscratch/sd/a/ahhyun/EcoGFound/PODCAST/podcast-benchmark/notebooks/results/supersubject/primary_metric_matrix_all_levels.csv


## Generate Per-Task Full Metric CSVs

For each task, produce a CSV with all metrics at multiple aggregation levels.

In [50]:
for task_name in all_tasks:
    print(f'\n{"="*60}')
    print(f'Task: {task_name}')
    print(f'Primary metric: {PRIMARY_METRIC[task_name]}')
    print(f'{"="*60}')

    task_sheets = {}

    for model in all_models:
        df = results_raw.get(model, {}).get(task_name)
        if df is None:
            continue

        # Level 1: Raw lag x fold
        raw = build_raw_lag_fold(df, split='test')
        if raw is not None:
            raw.insert(0, 'model', model)
            task_sheets.setdefault('raw_lag_fold', []).append(raw)

        # Level 2: Fold-averaged (per lag)
        fold_avg = build_fold_averaged(df, split='test')
        if fold_avg is not None:
            fold_avg.insert(0, 'model', model)
            task_sheets.setdefault('fold_averaged', []).append(fold_avg)

        # Level 3: Lag-averaged (per fold)
        lag_avg = build_lag_averaged(df, split='test')
        if lag_avg is not None:
            lag_avg.insert(0, 'model', model)
            task_sheets.setdefault('lag_averaged', []).append(lag_avg)

        # Level 4: Fully averaged
        full_avg = build_fully_averaged(df, split='test')
        if full_avg is not None:
            row = full_avg['mean'].to_frame().T.reset_index(drop=True)
            row_std = full_avg['std'].to_frame().T.add_suffix('_std').reset_index(drop=True)
            row = pd.concat([row, row_std], axis=1)
            row.insert(0, 'model', model)
            task_sheets.setdefault('fully_averaged', []).append(row)

        # Level 5: Best lag
        best = build_best_lag(df, task_name, split='test')
        if best is not None:
            row = best.to_frame().T
            row.insert(0, 'model', model)
            task_sheets.setdefault('best_lag', []).append(row)

    # Concatenate and save each level
    task_dir_name = TASK_NAME_TO_DIR.get(task_name, task_name)

    for level_name, dfs in task_sheets.items():
        combined = pd.concat(dfs, ignore_index=True)
        out_path = OUTPUT_DIR / f'{task_dir_name}_{level_name}.csv'
        combined.to_csv(out_path, index=False)
        print(f'  Saved: {out_path.name} ({combined.shape})')

        # Show preview for fully_averaged and best_lag
        if level_name in ('fully_averaged', 'best_lag'):
            display(combined)

print(f'\nAll per-task CSVs saved to {OUTPUT_DIR}')


Task: content_noncontent_task
Primary metric: roc_auc
  Saved: content_noncontent_fold_averaged.csv ((22, 20))
  Saved: content_noncontent_best_lag.csv ((4, 20))


,model,best_lag,bce_mean,bce_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std,acc_logits_mean,acc_logits_std,bce_with_logits_mean,bce_with_logits_std,f1_logits_mean,f1_logits_std,precision_logits_mean,precision_logits_std,sensitivity_logits_mean,sensitivity_logits_std,specificity_logits_mean,specificity_logits_std
0,baseline,200.0,0.714476,0.051342,0.558654,0.014646,0.590047,0.017606,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,brainbert,-250.0,NaN,NaN,NaN,NaN,0.524802,0.021495,0.457837,0.000496,0.900326,0.029208,0.323280,0.004686,0.017857,0.017857,0.007606,0.007606,0.934193,0.004960
2,diver,0.0,NaN,NaN,NaN,NaN,0.532903,0.030258,0.542163,0.002480,0.691735,0.000203,0.460015,0.045314,0.516369,0.025298,0.700066,0.261905,0.319775,0.297288
3,popt,-250.0,NaN,NaN,NaN,NaN,0.526951,0.038194,0.534722,0.000992,0.687179,0.001961,0.433617,0.002967,0.542824,0.003472,0.907242,0.001488,0.069279,0.004134


  Saved: content_noncontent_raw_lag_fold.csv ((24, 10))
  Saved: content_noncontent_lag_averaged.csv ((6, 9))
  Saved: content_noncontent_fully_averaged.csv ((3, 15))


,model,acc_logits,bce_with_logits,f1_logits,precision_logits,roc_auc,sensitivity_logits,specificity_logits,acc_logits_std,bce_with_logits_std,f1_logits_std,precision_logits_std,roc_auc_std,sensitivity_logits_std,specificity_logits_std
0,brainbert,0.495412,0.822255,0.400751,0.338955,0.499297,0.486607,0.466270,0.038283,0.098971,0.071305,0.267469,0.032909,0.431495,0.417653
1,diver,0.521081,0.694201,0.416329,0.414517,0.517816,0.607515,0.376075,0.039401,0.004831,0.058581,0.208106,0.025381,0.422771,0.410571
2,popt,0.540799,0.689068,0.428049,0.545759,0.518105,0.935764,0.046214,0.011839,0.002290,0.014751,0.006492,0.029208,0.058035,0.047170



Task: gpt_surprise_multiclass_task
Primary metric: roc_auc_multiclass
  Saved: gpt_surprise_multiclass_fold_averaged.csv ((22, 10))
  Saved: gpt_surprise_multiclass_best_lag.csv ((4, 10))


...


Task: word_embedding_decoding_task
Primary metric: pairwise_accuracy
  Saved: word_embedding_raw_lag_fold.csv ((50, 18))
  Saved: word_embedding_fold_averaged.csv ((10, 32))
  Saved: word_embedding_lag_averaged.csv ((5, 17))
  Saved: word_embedding_fully_averaged.csv ((1, 31))


,model,cosine_sim,mse,nll_embedding,occurence_perplexity,occurence_top_1,occurence_top_10,occurence_top_5,pairwise_accuracy,word_avg_auc_roc,word_perplexity,word_test_weighted_auc_roc,word_top_1,word_top_10,word_top_5,word_train_weighted_auc_roc,cosine_sim_std,mse_std,nll_embedding_std,occurence_perplexity_std,occurence_top_1_std,occurence_top_10_std,occurence_top_5_std,pairwise_accuracy_std,word_avg_auc_roc_std,word_perplexity_std,word_test_weighted_auc_roc_std,word_top_1_std,word_top_10_std,word_top_5_std,word_train_weighted_auc_roc_std
0,baseline,0.614597,0.344096,3.471956,1015.315131,0.001202,0.011701,0.006047,0.502461,0.566456,433.603417,0.562116,0.005693,0.045739,0.023579,0.564904,0.015001,0.006587,0.004912,0.471936,0.000918,0.003061,0.00264,0.006457,0.036675,11.745313,0.029281,0.002289,0.009578,0.006993,0.032429


  Saved: word_embedding_best_lag.csv ((1, 32))


,model,best_lag,cosine_sim_mean,cosine_sim_std,mse_mean,mse_std,nll_embedding_mean,nll_embedding_std,occurence_perplexity_mean,occurence_perplexity_std,occurence_top_1_mean,occurence_top_1_std,occurence_top_10_mean,occurence_top_10_std,occurence_top_5_mean,occurence_top_5_std,pairwise_accuracy_mean,pairwise_accuracy_std,word_avg_auc_roc_mean,word_avg_auc_roc_std,word_perplexity_mean,word_perplexity_std,word_test_weighted_auc_roc_mean,word_test_weighted_auc_roc_std,word_top_1_mean,word_top_1_std,word_top_10_mean,word_top_10_std,word_top_5_mean,word_top_5_std,word_train_weighted_auc_roc_mean,word_train_weighted_auc_roc_std
0,baseline,400.0,0.614586,0.013204,0.341903,0.006591,3.464171,0.001084,1015.190625,0.401361,0.001379,0.000788,0.012411,0.003024,0.008077,0.00349,0.511619,0.003706,0.604161,0.037197,433.358453,11.464188,0.597662,0.01721,0.007289,0.001005,0.052205,0.007122,0.030534,0.008209,0.601213,0.025003



All per-task CSVs saved to /pscratch/sd/a/ahhyun/EcoGFound/PODCAST/podcast-benchmark/notebooks/results/supersubject


## Summary: File Inventory

In [51]:
print('Generated files:')
print()
for f in sorted(OUTPUT_DIR.glob('*.csv')):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:60s}  {size_kb:6.1f} KB')

Generated files:

  content_noncontent_best_lag.csv                                  1.3 KB
  content_noncontent_fold_averaged.csv                             4.9 KB
  content_noncontent_fully_averaged.csv                            1.1 KB
  content_noncontent_lag_averaged.csv                              0.9 KB
  content_noncontent_raw_lag_fold.csv                              3.5 KB
  gpt_surprise_best_lag.csv                                        0.5 KB
  gpt_surprise_fold_averaged.csv                                   2.4 KB
  gpt_surprise_fully_averaged.csv                                  0.4 KB
  gpt_surprise_lag_averaged.csv                                    0.4 KB
  gpt_surprise_multiclass_best_lag.csv                             0.7 KB
  gpt_surprise_multiclass_fold_averaged.csv                        3.0 KB
  gpt_surprise_multiclass_fully_averaged.csv                       0.6 KB
  gpt_surprise_multiclass_lag_averaged.csv                         0.6 KB
  gpt_surprise_multi

## Source Paths Reference

Which result file was used for each model/task combination.

In [52]:
paths_df = pd.DataFrame(
    [(model, task, path) for (model, task), path in sorted(result_paths.items())],
    columns=['model', 'task', 'source_path']
)
display(paths_df)
paths_df.to_csv(OUTPUT_DIR / 'source_paths.csv', index=False)
print(f'Saved: {OUTPUT_DIR / "source_paths.csv"}')

,model,task,source_path
0,baseline,content_noncontent_task,/pscratch/sd/a/ahhyun/EcoGFound/PODCAST/podcas...
1,baseline,gpt_surprise_multiclass_task,/pscratch/sd/a/ahhyun/EcoGFound/PODCAST/podcas...
2,baseline,gpt_surprise_task,/pscratch/sd/a/ahhyun/EcoGFound/PODCAST/podcas...
3,baseline,iu_boundary_task,/pscratch/sd/a/ahhyun/EcoGFound/PODCAST/podcas...
4,baseline,llm_decoding_task,/pscratch/sd/a/ahhyun/EcoGFound/PODCAST/podcas...
5,baseline,pos_task,/pscratch/sd/a/ahhyun/EcoGFound/PODCAST/podcas...
6,baseline,sentence_onset_task,/pscratch/sd/a/ahhyun/EcoGFound/PODCAST/podcas...
7,baseline,volume_level_decoding_task,/pscratch/sd/a/ahhyun/EcoGFound/PODCAST/podcas...
8,baseline,whisper_embedding_decoding_task,/pscratch/sd/a/ahhyun/EcoGFound/PODCAST/podcas...
9,baseline,word_embedding_decoding_task,/pscratch/sd/a/ahhyun/EcoGFound/PODCAST/podcas...


Saved: /pscratch/sd/a/ahhyun/EcoGFound/PODCAST/podcast-benchmark/notebooks/results/supersubject/source_paths.csv
